# QSense NN Ablation Training on Kaggle GPU

**Instructions:**
This trains the 12-feature ablation network (ignoring dispersion features).
1. Upload the full QSense repo as a Kaggle Dataset.
2. Select a GPU accelerator (T4 or P100).
3. Run all cells.
4. Download rtifacts/nn_weights_ablation.npz and rtifacts/nn_meta_ablation.json.


In [ ]:
# If running from Kaggle with the repo as a dataset, adjust the path below
import sys, os

# Detect Kaggle environment
if os.path.exists('/kaggle/input'):
    # Find the repo directory in Kaggle input
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train_nn.py' in files or 'kaggle_train.py' in files:
            # Navigate up to repo root if needed
            if 'train' in dirs or any(f.endswith('.py') for f in files):
                repo_root = root
                break
    else:
        # Fallback: copy files
        repo_root = '/kaggle/working'
    print(f'Repo root: {repo_root}')
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    os.chdir(repo_root)
else:
    # Local run
    repo_root = os.getcwd()
    print(f'Local run from: {repo_root}')

print('Python path:', sys.path[:3])

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'],
                       capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else 'torch already installed')

In [ ]:
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from pathlib import Path

# Create artifacts directory in working directory
artifacts_dir = Path('/kaggle/working/artifacts') if os.path.exists('/kaggle') else Path('artifacts')
artifacts_dir.mkdir(parents=True, exist_ok=True)
print(f'Artifacts will be saved to: {artifacts_dir}')

In [ ]:
from train.train_ablation import train

model, best_val_nll = train(
    steps      = 20_000,
    batch      = 2048,
    lr         = 1e-3,
    lr_min     = 1e-5,
    val_every  = 500,
    out        = artifacts_dir / 'nn_weights_ablation.npz',
    seed       = 20261004,
    device_str = 'auto',
)
print(f'\\nTraining complete. Best validation NLL: {best_val_nll:.4f}')

In [ ]:
# Skip inference check for ablation.

In [ ]:
# Skip torch match check for ablation.

In [ ]:
# List output files
for f in sorted(artifacts_dir.glob('*')):
    print(f'{f.name:30} {f.stat().st_size / 1024:.1f} KB')